# German Train Delay Dataset: Data Pipeline

Builds the train and test datasets used for the delay-prediction models, from raw Deutsche Bahn data to weather, infrastructure, and timetable-enriched snapshot files.
Scope: Berlin, Hamburg, and Munich metro areas. Train period is November 2025 to April 2026. Test period is May to July 2026 (three months, forward-in-time).

In [4]:
import os
import re
import time
import json
import calendar
from datetime import timedelta, datetime, date

import polars as pl
import numpy as np
import requests
from huggingface_hub import hf_hub_download
from shapely import wkt
from shapely.geometry import Point
from tqdm import tqdm

## Configuration

`TRAIN_MONTHS` and `TEST_MONTHS` are consecutive and non-overlapping, so the test set is always forward-in-time relative to training. Full-nationwide-coverage data only starts 2025-11-02, so November 2025 is the earliest usable month.


`MAX_FUTURE_HORIZON_MINUTES` and `MAX_PAST_HORIZON_MINUTES` cap how far in elapsed time a past or future stop can be from a snapshot before it is excluded or clipped. Both were added after real bugs: a long-tail of trip segments (predominantly Munich S-Bahn) had "future stops" that were, in real elapsed time, days away rather than minutes, which caused catastrophic extrapolation in the gradient-based models. The future-side fix excludes the pathological stop entirely; the past-side fix clips the elapsed-time value while leaving `last_known_delay`/`past_delay_i` intact, since those should reflect the true most recent stop regardless of how long ago it was.

Every stage's random sampling (`sample_snapshots`) uses a fixed seed, so given the same raw input data this notebook produces byte-identical output on every run.

In [3]:
LOCAL_DATA_DIR = "./dbdata"

TRAIN_MONTHS = ["2025-11", "2025-12", "2026-01", "2026-02", "2026-03", "2026-04"]
TEST_MONTHS = ["2026-05", "2026-06", "2026-07"]
ALL_MONTHS = TRAIN_MONTHS + TEST_MONTHS

N_SNAPSHOTS_PER_TRAIN_MONTH = 2000
N_SNAPSHOTS_PER_TEST_MONTH = 400

CITY_PREFIXES = ["Berlin", "Hamburg", "München"]

N_FUTURE_STOPS = 10
MAX_FUTURE_HORIZON_MINUTES = 360
MAX_PAST_HORIZON_MINUTES = 360
IDLE_BEG_MIN = 5
IDLE_END_MIN = 5
N_PAST_HISTORY = 3

STATION_CACHE_PATH = f"{LOCAL_DATA_DIR}/station_coordinates.json"

WEATHER_CACHE_PATH = f"{LOCAL_DATA_DIR}/weather_lookup_openmeteo.parquet"
WEATHER_PROGRESS_PATH = f"{LOCAL_DATA_DIR}/weather_fetch_progress.json"

TRAIN_SPAN_LABEL = f"{TRAIN_MONTHS[0]}_to_{TRAIN_MONTHS[-1]}"
TEST_SPAN_LABEL = f"{TEST_MONTHS[0]}_to_{TEST_MONTHS[-1]}"
TRAIN_FINAL_PATH = f"{LOCAL_DATA_DIR}/train_3city_{TRAIN_SPAN_LABEL}_weather.parquet"
TEST_FINAL_PATH = f"{LOCAL_DATA_DIR}/test_3city_{TEST_SPAN_LABEL}_weather.parquet"

OPEN_METEO_URL = "https://archive-api.open-meteo.com/v1/archive"
HOURLY_WEATHER_VARS = ["temperature_2m", "precipitation", "wind_speed_10m"]

NOMINATIM_URL = "https://nominatim.openstreetmap.org/search"
USER_AGENT = "GISMA-MSc-Thesis-TrainDelayPrediction/1.0 (research project)"
NOMINATIM_RATE_LIMIT_SECONDS = 1.1
OPEN_METEO_RATE_LIMIT_SECONDS = 0.2

## Stage 1: Download raw monthly data

Each month of German rail data is downloaded once from the `piebro/deutsche-bahn-data` Hugging Face dataset and cached locally as `germany_{month}.parquet`.

In [4]:
def download_month(month):
    os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
    local_path = os.path.join(LOCAL_DATA_DIR, f"germany_{month}.parquet")
    if os.path.isfile(local_path):
        print(f"[checkpoint] {local_path} already exists, skipping download.")
        return local_path

    fname = f"monthly_processed_data/data-{month}.parquet"
    print(f"Downloading {fname}...")
    downloaded_path = hf_hub_download(
        repo_id="piebro/deutsche-bahn-data", repo_type="dataset",
        filename=fname, local_dir=LOCAL_DATA_DIR,
    )
    if downloaded_path != local_path:
        os.rename(downloaded_path, local_path)
    size_mb = os.path.getsize(local_path) / (1024 * 1024)
    print(f"Downloaded: {local_path} ({size_mb:.1f} MB)")
    return local_path


def filter_to_cities(df, station_col="station_name", city_prefixes=CITY_PREFIXES):
    pattern = "|".join(f"^{re.escape(p)}" for p in city_prefixes)
    return df.filter(pl.col(station_col).str.contains(pattern))

## Stage 2: Trip segmentation

`train_line_ride_id` in the raw data is reused across every calendar day a recurring service runs, and it does not uniquely identify a single journey. Left uncorrected, this causes every recurring day of a service to be treated as one continuous multi-day trip.

The fix sorts each ride's stops chronologically and starts a new trip segment whenever the station sequence number resets or goes backwards.

`compute_activity_windows` then gives each trip segment an appear/disappear time window (with a small idle buffer before departure and after arrival),

`get_active_rides` answers "which trip segments are active at this exact snapshot time", the basis for every snapshot built in Stage 3.

In [5]:
def add_trip_segment_id(df):
    df = df.with_columns(
        pl.coalesce([pl.col("departure_change_time"), pl.col("arrival_change_time")]).alias("stop_time")
    )
    df = df.sort(["train_line_ride_id", "stop_time"])

    prev_ride = pl.col("train_line_ride_id").shift(1)
    prev_num = pl.col("train_line_station_num").shift(1)
    is_new_segment = (
        (pl.col("train_line_ride_id") != prev_ride)
        | (pl.col("train_line_station_num") <= prev_num)
        | prev_num.is_null()
    )
    df = df.with_columns(is_new_segment.fill_null(True).alias("_is_new_segment"))
    df = df.with_columns(pl.col("_is_new_segment").cum_sum().alias("trip_segment_id"))
    return df.drop("_is_new_segment")


def compute_activity_windows(df):
    per_journey = (
        df.sort(["trip_segment_id", "train_line_station_num"])
        .group_by("trip_segment_id")
        .agg([
            pl.col("train_line_ride_id").first(),
            pl.col("departure_planned_time").first().alias("first_planned_dep"),
            pl.col("departure_change_time").first().alias("first_observed_dep"),
            pl.col("arrival_planned_time").last().alias("last_planned_arr"),
            pl.col("arrival_change_time").last().alias("last_observed_arr"),
        ])
    )
    per_journey = per_journey.with_columns([
        pl.min_horizontal([
            (pl.col("first_planned_dep") - timedelta(minutes=IDLE_BEG_MIN)),
            pl.col("first_observed_dep"),
        ]).alias("appear_time"),
        (pl.coalesce([pl.col("last_observed_arr"), pl.col("last_planned_arr")])
         + timedelta(minutes=IDLE_END_MIN)).alias("disappear_time"),
    ])
    return per_journey.select(["trip_segment_id", "train_line_ride_id", "appear_time", "disappear_time"])


def get_active_rides(activity_windows, snapshot_time):
    return activity_windows.filter(
        (pl.col("appear_time") <= snapshot_time) & (pl.col("disappear_time") > snapshot_time)
    )["trip_segment_id"]

## Stage 3: Snapshot and feature construction

For each snapshot time, every active trip segment is split into its past (already happened) and future (not yet happened) stops. `last_known_delay` and `past_delay_i` come from the past side; the prediction targets, `future_delay_i`, come from the future side.

`compute_network_congestion` computes, for every active trip at this snapshot, how many *other* trips currently share its current station and their average delay right now. 

In [6]:
def compute_network_congestion(active_df, snapshot_time):
    current_positions = []
    for trip_id, group in active_df.group_by("trip_segment_id"):
        past = group.filter(pl.col("stop_time") <= snapshot_time).sort("train_line_station_num")
        if len(past) == 0:
            continue
        current_positions.append({
            "trip_segment_id": trip_id[0] if isinstance(trip_id, tuple) else trip_id,
            "current_station": past["station_name"][-1],
            "current_delay": past["delay_in_min"][-1],
        })

    if not current_positions:
        return {}, {}

    pos_df = pl.DataFrame(current_positions)
    station_stats = pos_df.group_by("current_station").agg([
        pl.len().alias("n_trains_at_station"),
        pl.col("current_delay").mean().alias("avg_delay_at_station"),
    ])
    station_stats_lookup = {
        row["current_station"]: (row["n_trains_at_station"], row["avg_delay_at_station"])
        for row in station_stats.iter_rows(named=True)
    }

    result = {}
    pos_with_stats = pos_df.join(station_stats, on="current_station")
    for row in pos_with_stats.iter_rows(named=True):
        n_others = row["n_trains_at_station"] - 1
        if n_others > 0:
            avg_others = ((row["avg_delay_at_station"] * row["n_trains_at_station"]) - row["current_delay"]) / n_others
        else:
            avg_others = 0.0
        result[row["trip_segment_id"]] = (row["current_station"], n_others, avg_others)
    return result, station_stats_lookup

`build_snapshot_targets` builds one row per active trip segment at a given snapshot.
- **Future-horizon cap.** A small, station-concentrated tail of trip segments (predominantly Munich S-Bahn) has a "next stop" that is, in real elapsed time, up to around 15 days away, against a normal median of about 15 minutes at that horizon. `MAX_FUTURE_HORIZON_MINUTES` (360, six hours) excludes these stops entirely, which simply reduces the number of valid future targets for that trip.

In [7]:
def build_snapshot_targets(df, trip_ids, snapshot_time, n_future=N_FUTURE_STOPS, total_filtered_by_horizon_cap=None):
    if total_filtered_by_horizon_cap is None:
        total_filtered_by_horizon_cap = [0]

    active_df = df.filter(pl.col("trip_segment_id").is_in(trip_ids.to_list()))
    active_df = active_df.sort(["trip_segment_id", "train_line_station_num"])
    active_df = active_df.with_columns(
        pl.coalesce([pl.col("departure_change_time"), pl.col("arrival_change_time")]).alias("stop_time")
    )

    congestion, station_stats_lookup = compute_network_congestion(active_df, snapshot_time)

    rows = []
    for trip_id, group in active_df.group_by("trip_segment_id"):
        group = group.sort("train_line_station_num")
        past = group.filter(pl.col("stop_time") <= snapshot_time)
        future = group.filter(pl.col("stop_time") > snapshot_time)
        if len(past) == 0 or len(future) == 0:
            continue

        last_known_delay = past["delay_in_min"][-1]
        past_delays_recent_first = past["delay_in_min"].to_list()[::-1][:N_PAST_HISTORY]
        past_times_recent_first = past["stop_time"].to_list()[::-1][:N_PAST_HISTORY]
        past_minutes_ago_raw = [
            (snapshot_time - pt).total_seconds() / 60.0 if pt is not None else None
            for pt in past_times_recent_first
        ]
        past_minutes_ago = [
            min(v, MAX_PAST_HORIZON_MINUTES) if v is not None else None
            for v in past_minutes_ago_raw
        ]

        n_before_horizon_filter = len(future)
        future = future.filter(
            (pl.col("stop_time") - snapshot_time).dt.total_minutes() <= MAX_FUTURE_HORIZON_MINUTES
        )
        total_filtered_by_horizon_cap[0] += n_before_horizon_filter - len(future)

        future_delays = future["delay_in_min"].to_list()[:n_future]
        future_stations = future["station_name"].to_list()[:n_future]
        future_times = future["stop_time"].to_list()[:n_future]
        future_minutes_ahead = [
            (ft - snapshot_time).total_seconds() / 60.0 if ft is not None else None
            for ft in future_times
        ]
        future_station_congestion_n = [
            station_stats_lookup.get(fs, (0, 0.0))[0] if fs is not None else None
            for fs in future_stations
        ]
        future_station_congestion_avgdelay = [
            station_stats_lookup.get(fs, (0, 0.0))[1] if fs is not None else None
            for fs in future_stations
        ]

        this_trip_id = trip_id[0] if isinstance(trip_id, tuple) else trip_id
        current_station, n_others_at_station, avg_delay_others = congestion.get(this_trip_id, (None, 0, 0.0))

        row = {
            "trip_segment_id": this_trip_id,
            "train_line_ride_id": group["train_line_ride_id"][0],
            "snapshot_time": snapshot_time,
            "last_known_delay": last_known_delay,
            "n_valid_future_targets": len(future_delays),
            "train_type": group["train_type"][0] if "train_type" in group.columns else None,
            "snapshot_hour": snapshot_time.hour,
            "snapshot_dow": snapshot_time.weekday(),
            "current_station": current_station,
            "n_trains_at_current_station": n_others_at_station,
            "avg_delay_others_at_current_station": avg_delay_others,
        }
        for i in range(1, N_PAST_HISTORY + 1):
            row[f"past_delay_{i}"] = past_delays_recent_first[i-1] if i <= len(past_delays_recent_first) else None
            row[f"past_minutes_ago_{i}"] = past_minutes_ago[i-1] if i <= len(past_minutes_ago) else None
        for i in range(n_future):
            row[f"future_delay_{i+1}"] = future_delays[i] if i < len(future_delays) else None
            row[f"future_station_{i+1}"] = future_stations[i] if i < len(future_stations) else None
            row[f"future_minutes_ahead_{i+1}"] = future_minutes_ahead[i] if i < len(future_minutes_ahead) else None
            row[f"future_station_congestion_n_{i+1}"] = future_station_congestion_n[i] if i < len(future_station_congestion_n) else None
            row[f"future_station_congestion_avgdelay_{i+1}"] = future_station_congestion_avgdelay[i] if i < len(future_station_congestion_avgdelay) else None
        rows.append(row)

    if not rows:
        return pl.DataFrame()

    schema = {
        "trip_segment_id": pl.UInt32, "train_line_ride_id": pl.Utf8, "snapshot_time": pl.Datetime,
        "last_known_delay": pl.Int64, "n_valid_future_targets": pl.Int64,
        "train_type": pl.Utf8, "snapshot_hour": pl.Int64, "snapshot_dow": pl.Int64,
        "current_station": pl.Utf8,
        "n_trains_at_current_station": pl.Int64, "avg_delay_others_at_current_station": pl.Float64,
    }
    for i in range(1, N_PAST_HISTORY + 1):
        schema[f"past_delay_{i}"] = pl.Int64
        schema[f"past_minutes_ago_{i}"] = pl.Float64
    for i in range(1, n_future + 1):
        schema[f"future_delay_{i}"] = pl.Int64
        schema[f"future_station_{i}"] = pl.Utf8
        schema[f"future_minutes_ahead_{i}"] = pl.Float64
        schema[f"future_station_congestion_n_{i}"] = pl.Int64
        schema[f"future_station_congestion_avgdelay_{i}"] = pl.Float64

    return pl.DataFrame(rows, schema=schema)

`sample_snapshots` picks a random set of real stop-event timestamps to use as snapshot times for a given month. `build_training_dataset` ties Stages 2 and 3 together for one month: segment trips, compute activity windows, sample snapshot times, then build target rows for each one. `build_snapshot_dataset_if_needed` uses checkpoint: if the month's output file already exists, load it instead of rebuilding.

In [8]:
def sample_snapshots(df, n_snapshots, seed=42):
    valid_times = df.select(
        pl.coalesce([pl.col("departure_change_time"), pl.col("arrival_change_time")])
    ).drop_nulls().to_series().unique()
    n = min(n_snapshots, len(valid_times))
    return valid_times.sample(n, seed=seed, with_replacement=False).sort()


def build_training_dataset(df, n_snapshots, n_future=N_FUTURE_STOPS, seed=42, label=""):
    df = add_trip_segment_id(df)
    windows = compute_activity_windows(df)
    snapshot_times = sample_snapshots(df, n_snapshots, seed=seed)

    total_filtered_by_horizon_cap = [0]
    all_rows = []
    for i, snap_time in enumerate(snapshot_times):
        active = get_active_rides(windows, snap_time)
        targets = build_snapshot_targets(df, active, snap_time, n_future, total_filtered_by_horizon_cap)
        if len(targets) > 0:
            all_rows.append(targets)
        if (i + 1) % max(1, len(snapshot_times) // 10) == 0:
            print(f"  [{label}] {i+1}/{len(snapshot_times)} snapshots processed")

    if not all_rows:
        return pl.DataFrame()
    if total_filtered_by_horizon_cap[0] > 0:
        print(f"  [{label}] future-horizon cap filtered {total_filtered_by_horizon_cap[0]:,} future stops")
    return pl.concat(all_rows, how="vertical_relaxed")


def month_dataset_path(month, label):
    return f"{LOCAL_DATA_DIR}/{label}_{month}_snap.parquet"


def build_snapshot_dataset_if_needed(month, out_path, n_snapshots, label):
    if os.path.isfile(out_path):
        print(f"[checkpoint] {out_path} already exists, skipping.")
        return pl.read_parquet(out_path)

    raw_path = download_month(month)
    df = pl.read_parquet(raw_path)
    df = filter_to_cities(df)

    dataset = build_training_dataset(df, n_snapshots, label=label)
    dataset.write_parquet(out_path)
    print(f"Saved {label}: {len(dataset):,} rows -> {out_path}")
    return dataset

Building snapshot dataset for every train and test month, and combine each group into one training set and one test set.

In [10]:
train_month_datasets = []
for month in TRAIN_MONTHS:
    out_path = month_dataset_path(month, "train")
    train_month_datasets.append(build_snapshot_dataset_if_needed(month, out_path, N_SNAPSHOTS_PER_TRAIN_MONTH, f"train-{month}"))

test_month_datasets = []
for month in TEST_MONTHS:
    out_path = month_dataset_path(month, "test")
    test_month_datasets.append(build_snapshot_dataset_if_needed(month, out_path, N_SNAPSHOTS_PER_TEST_MONTH, f"test-{month}"))

train_dataset = pl.concat(train_month_datasets, how="vertical_relaxed")
test_dataset = pl.concat(test_month_datasets, how="vertical_relaxed")
print(f"Combined training set: {len(train_dataset):,} rows")
print(f"Combined test set: {len(test_dataset):,} rows")

[checkpoint] ./dbdata\germany_2025-11.parquet already exists, skipping download.
  [train-2025-11] 200/2000 snapshots processed
  [train-2025-11] 400/2000 snapshots processed
  [train-2025-11] 600/2000 snapshots processed
  [train-2025-11] 800/2000 snapshots processed
  [train-2025-11] 1000/2000 snapshots processed
  [train-2025-11] 1200/2000 snapshots processed
  [train-2025-11] 1400/2000 snapshots processed
  [train-2025-11] 1600/2000 snapshots processed
  [train-2025-11] 1800/2000 snapshots processed
  [train-2025-11] 2000/2000 snapshots processed
  [train-2025-11] future-horizon cap filtered 61,401 future stops
Saved train-2025-11: 409,226 rows -> ./dbdata/train_2025-11_snap.parquet
[checkpoint] ./dbdata\germany_2025-12.parquet already exists, skipping download.
  [train-2025-12] 200/2000 snapshots processed
  [train-2025-12] 400/2000 snapshots processed
  [train-2025-12] 600/2000 snapshots processed
  [train-2025-12] 800/2000 snapshots processed
  [train-2025-12] 1000/2000 snapsho

## Stage 4: Geocoding stations

Every station name that appears as a *future* stop needs coordinates, both for the weather fetch below and for the infrastructure matching later. 

Geocoding uses OpenStreetMap's Nominatim, with a plain station-name query

In [11]:
station_cols = {f"future_station_{i}" for i in range(1, N_FUTURE_STOPS + 1)}
needed_stations = set()
for df in [train_dataset, test_dataset]:
    for c in [col for col in df.columns if col in station_cols]:
        needed_stations.update(df[c].drop_nulls().unique().to_list())
needed_stations = sorted(s for s in needed_stations if s is not None)
print(f"This run needs {len(needed_stations)} unique stations.")

This run needs 183 unique stations.


In [12]:
def geocode_station(station_name, debug=False):
    headers = {"User-Agent": USER_AGENT}
    attempts = [
        {"q": station_name, "countrycodes": "de", "format": "json", "limit": 1},
        {"q": f"{station_name}, Germany", "format": "json", "limit": 1},
    ]
    for attempt_i, params in enumerate(attempts):
        resp = requests.get(NOMINATIM_URL, params=params, headers=headers, timeout=10)
        if debug:
            print(f"    [debug] attempt {attempt_i+1} status={resp.status_code} url={resp.url}")
        resp.raise_for_status()
        results = resp.json()
        if results:
            return {"lat": float(results[0]["lat"]), "lon": float(results[0]["lon"])}
        time.sleep(NOMINATIM_RATE_LIMIT_SECONDS)
    return None

In [ ]:
cache = {}
if os.path.isfile(STATION_CACHE_PATH):
    with open(STATION_CACHE_PATH, encoding="utf-8") as f:
        cache = json.load(f)
    n_success = sum(1 for v in cache.values() if v is not None)
    print(f"[checkpoint] Loaded {len(cache)} cached entries ({n_success} succeeded)")

to_geocode = [s for s in needed_stations if cache.get(s) is None]
print(f"{len(to_geocode)}/{len(needed_stations)} stations need geocoding")

for i, station in enumerate(to_geocode):
    cache[station] = geocode_station(station, debug=(i == 0))
    if (i + 1) % 10 == 0:
        print(f"  geocoded {i+1}/{len(to_geocode)}")
        with open(STATION_CACHE_PATH, "w", encoding="utf-8") as f:
            json.dump(cache, f)
    time.sleep(NOMINATIM_RATE_LIMIT_SECONDS)

with open(STATION_CACHE_PATH, "w", encoding="utf-8") as f:
    json.dump(cache, f)
n_found = sum(1 for v in cache.values() if v is not None)
print(f"Geocoded {n_found}/{len(cache)} stations successfully.")

station_coords = {k: v for k, v in cache.items() if v is not None}

[checkpoint] Loaded 183 cached entries (183 succeeded)
Geocoded 183/183 stations successfully.


## Stage 5: Fetch weather (Open-Meteo)

Weather is fetched from Open-Meteo's historical archive API (a separate subdomain from its live forecast API), one station at a time, for each calendar month covering the full train and test span.

In [5]:
def fetch_weather_for_coords(lat, lon, date_range, debug=False):
    params = {
        "latitude": lat, "longitude": lon,
        "start_date": date_range[0], "end_date": date_range[1],
        "hourly": ",".join(HOURLY_WEATHER_VARS),
        "timezone": "Europe/Berlin",
    }
    resp = requests.get(OPEN_METEO_URL, params=params, timeout=15)
    if debug:
        print(f"    [debug] status={resp.status_code} url={resp.url}")
    resp.raise_for_status()
    data = resp.json()
    hourly = data["hourly"]
    return pl.DataFrame({
        "weather_hour": [datetime.fromisoformat(t) for t in hourly["time"]],
        **{var: hourly[var] for var in HOURLY_WEATHER_VARS if var in hourly},
    })

In [6]:
def end_of_month_str(month):
    year, mon = (int(x) for x in month.split("-"))
    last_day = calendar.monthrange(year, mon)[1]
    return f"{month}-{last_day:02d}"

WEATHER_DATE_RANGES = {month: (f"{month}-01", end_of_month_str(month)) for month in ALL_MONTHS}

In [9]:
all_rows = []
completed = set()

if os.path.isfile(WEATHER_PROGRESS_PATH) and os.path.isfile(WEATHER_CACHE_PATH):
    with open(WEATHER_PROGRESS_PATH, encoding="utf-8") as f:
        completed = set(tuple(x) for x in json.load(f))
    all_rows.append(pl.read_parquet(WEATHER_CACHE_PATH))
    print(f"[checkpoint] Resuming: {len(completed)} (station, month) pairs already fetched.")

if os.path.isfile(WEATHER_CACHE_PATH) and not os.path.isfile(WEATHER_PROGRESS_PATH):
    print(f"[checkpoint] {WEATHER_CACHE_PATH} already exists, loading from cache.")
    weather_wide = pl.read_parquet(WEATHER_CACHE_PATH)
    weather_wide = weather_wide.unique(subset=["train_station", "weather_hour"], keep="first")
else:
    first_debugged = len(completed) > 0
    for i, (station_name, coords) in enumerate(station_coords.items()):
        for period_name, date_range in WEATHER_DATE_RANGES.items():
            if (station_name, period_name) in completed:
                continue
            # print(f"[{i+1}/{len(station_coords)}] {station_name} / {period_name}")
            df = fetch_weather_for_coords(coords["lat"], coords["lon"], date_range, debug=(not first_debugged))
            first_debugged = True
            df = df.with_columns(pl.lit(station_name).alias("train_station"))
            all_rows.append(df)
            completed.add((station_name, period_name))

            combined_so_far = pl.concat(all_rows, how="vertical_relaxed")
            combined_so_far.write_parquet(WEATHER_CACHE_PATH)
            with open(WEATHER_PROGRESS_PATH, "w", encoding="utf-8") as f:
                json.dump(list(completed), f)
            time.sleep(OPEN_METEO_RATE_LIMIT_SECONDS)

    weather_wide = pl.concat(all_rows, how="vertical_relaxed")
    weather_wide = weather_wide.unique(subset=["train_station", "weather_hour"], keep="first")

print(f"Weather fetch complete: {len(weather_wide):,} rows, "
      f"{weather_wide['train_station'].n_unique()}/{len(station_coords)} stations covered.")

[checkpoint] Resuming: 1647 (station, month) pairs already fetched.
Weather fetch complete: 1,199,016 rows, 183/183 stations covered.


## Stage 6: Join weather into the snapshot datasets

For each future horizon, the target station's weather at the hour the train is due there (`snapshot_time + future_minutes_ahead_i`, truncated to the hour) is joined in. 


In [17]:
def join_weather_for_horizon(snapshot_lf, weather_wide, horizon_i):
    station_col = f"future_station_{horizon_i}"
    minutes_col = f"future_minutes_ahead_{horizon_i}"
    if station_col not in snapshot_lf.collect_schema().names() or minutes_col not in snapshot_lf.collect_schema().names():
        return snapshot_lf

    snapshot_lf = snapshot_lf.with_columns(
        (pl.col("snapshot_time") + pl.duration(minutes=pl.col(minutes_col)))
        .dt.truncate("1h").alias(f"_target_hour_{horizon_i}")
    )
    weather_renamed = weather_wide.rename({"train_station": station_col, "weather_hour": f"_target_hour_{horizon_i}"})
    weather_cols = [c for c in weather_renamed.columns if c not in (station_col, f"_target_hour_{horizon_i}")]
    weather_renamed = weather_renamed.rename({c: f"future_{c}_{horizon_i}" for c in weather_cols})
    joined = snapshot_lf.join(weather_renamed.lazy(), on=[station_col, f"_target_hour_{horizon_i}"], how="left")
    return joined.drop(f"_target_hour_{horizon_i}")

In [18]:
for label, snap_df, out_path in [
    ("train", train_dataset, TRAIN_FINAL_PATH),
    ("test", test_dataset, TEST_FINAL_PATH),
]:
    if os.path.isfile(out_path):
        print(f"[checkpoint] {out_path} already exists, skipping.")
        continue

    lf = snap_df.lazy()
    for i in range(1, N_FUTURE_STOPS + 1):
        lf = join_weather_for_horizon(lf, weather_wide, i)
    enriched = lf.collect(engine="streaming")

    enriched.write_parquet(out_path)
    print(f"Saved {label}: {out_path} ({len(enriched):,} rows, {len(enriched.columns)} columns)")

Saved train: ./dbdata/train_3city_2025-11_to_2026-04_weather.parquet (2,513,797 rows, 97 columns)
Saved test: ./dbdata/test_3city_2026-05_to_2026-07_weather.parquet (258,560 rows, 97 columns)


## InfraGO infrastructure extraction

Track infrastructure (number of tracks, line speed, electrification) comes from DB InfraGO's `Streckennetz` (track network) file, matched to each already-geocoded station by nearest spatial distance rather than by name. 

Two real parsing issues were found by inspecting the raw file directly rather than assuming a schema:

- **`Gleisanzahl`** (track count) is as text category (`eingleisig` = single-track, `zweigleisig` = double-track), plus a couple of "no data available" placeholder strings and occasional nulls.
- **`Geschwindigkeit`** (speed) is mostly `"{number} km/h"` text, plus two genuine non-numeric codes (`SKVerb`, `kein VZG erforderlich`, German for "no speed directory required")


A station's spatial match is capped at `MAX_MATCH_DISTANCE_DEGREES` (0.05); beyond that, a "nearest" segment is more likely a data gap than a real match.

In [19]:
STRECKENNETZ_PATH = "./dbdata/M1 Streckennetz.csv"
INFRA_OUTPUT_PATH = "./dbdata/station_infrastructure.json"
MAX_MATCH_DISTANCE_DEGREES = 0.05


def parse_gleisanzahl(value):
    if value is None:
        return None
    mapping = {"eingleisig": 1.0, "zweigleisig": 2.0}
    return mapping.get(value)


def parse_geschwindigkeit(value):
    if value is None or " km/h" not in value:
        return None
    return float(value.replace(" km/h", "").strip().replace(",", "."))

In [20]:
streckennetz = pl.read_csv(STRECKENNETZ_PATH, separator=";", encoding="utf8")
streckennetz = streckennetz.with_columns([
    pl.col("Gleisanzahl").map_elements(parse_gleisanzahl, return_dtype=pl.Float64).alias("gleisanzahl_parsed"),
    pl.col("Geschwindigkeit").map_elements(parse_geschwindigkeit, return_dtype=pl.Float64).alias("geschwindigkeit_parsed"),
])
print(f"Loaded {len(streckennetz):,} Streckennetz segments.")
print(f"gleisanzahl_parsed non-null: {streckennetz['gleisanzahl_parsed'].is_not_null().sum()}/{len(streckennetz)}")
print(f"geschwindigkeit_parsed non-null: {streckennetz['geschwindigkeit_parsed'].is_not_null().sum()}/{len(streckennetz)}")

Loaded 33,425 Streckennetz segments.
gleisanzahl_parsed non-null: 33389/33425
geschwindigkeit_parsed non-null: 32347/33425


In [21]:
geometries = []
n_failed = 0
for wkt_str in streckennetz["WKT WGS84 (EPSG 4326)"].to_list():
    try:
        geom = wkt.loads(wkt_str) if wkt_str else None
    except Exception:
        geom = None
        n_failed += 1
    geometries.append(geom)
if n_failed:
    print(f"{n_failed} geometries failed to parse and were skipped.")

Spatial matching is nearest-segment: at the scale of a few hundred stations, no spatial index.

In [22]:
def find_nearest_segment(station_point, segments_df, geometries):
    best_idx, best_dist = None, float("inf")
    for i, geom in enumerate(geometries):
        if geom is None:
            continue
        d = station_point.distance(geom)
        if d < best_dist:
            best_dist = d
            best_idx = i
    if best_idx is None or best_dist > MAX_MATCH_DISTANCE_DEGREES:
        return None
    return {
        "gleisanzahl": segments_df["gleisanzahl_parsed"][best_idx],
        "geschwindigkeit": segments_df["geschwindigkeit_parsed"][best_idx],
        "elektrifizierung": segments_df["Elektrifizierung"][best_idx],
        "distance_degrees": best_dist,
    }

In [23]:
infra_results = {}
n_matched, n_unmatched = 0, 0
for i, (station_name, coords) in enumerate(station_coords.items()):
    point = Point(coords["lon"], coords["lat"])
    match = find_nearest_segment(point, streckennetz, geometries)
    infra_results[station_name] = match
    n_matched += match is not None
    n_unmatched += match is None
    if (i + 1) % 20 == 0:
        print(f"  {i+1}/{len(station_coords)} stations matched")

print(f"Matched {n_matched}/{len(station_coords)} stations (within {MAX_MATCH_DISTANCE_DEGREES} degrees).")

with open(INFRA_OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(infra_results, f, indent=2)
print(f"Saved: {INFRA_OUTPUT_PATH}")

  20/183 stations matched
  40/183 stations matched
  60/183 stations matched
  80/183 stations matched
  100/183 stations matched
  120/183 stations matched
  140/183 stations matched
  160/183 stations matched
  180/183 stations matched
Matched 183/183 stations (within 0.05 degrees).
Saved: ./dbdata/station_infrastructure.json


## Join infrastructure features

The per-station infrastructure lookup just built is joined onto the weather-joined snapshot files, per horizon, producing `future_gleisanzahl_i`, `future_geschwindigkeit_i`, and `future_elektrifizierung_i` columns. Unmatched stations get null infra features, the same as any other missing-data case.

In [24]:
INFRA_TRAIN_IN = TRAIN_FINAL_PATH
INFRA_TEST_IN = TEST_FINAL_PATH
INFRA_TRAIN_OUT = f"{LOCAL_DATA_DIR}/train_3city_{TRAIN_SPAN_LABEL}_weather_infra.parquet"
INFRA_TEST_OUT = f"{LOCAL_DATA_DIR}/test_3city_{TEST_SPAN_LABEL}_weather_infra.parquet"

with open(INFRA_OUTPUT_PATH, encoding="utf-8") as f:
    raw_infra = json.load(f)

infra_rows = []
for station_name, match in raw_infra.items():
    if match is None:
        continue
    infra_rows.append({
        "station_name": station_name,
        "gleisanzahl": match.get("gleisanzahl"),
        "geschwindigkeit": match.get("geschwindigkeit"),
        "elektrifizierung": match.get("elektrifizierung"),
    })
infra_lookup = pl.DataFrame(infra_rows, schema={"station_name": pl.Utf8, "gleisanzahl": pl.Float64,
                                                 "geschwindigkeit": pl.Float64, "elektrifizierung": pl.Utf8})
print(f"Infrastructure lookup: {len(infra_lookup)} matched stations.")

Infrastructure lookup: 183 matched stations.


In [25]:
def join_infra_for_horizon(snapshot_lf, lookup, horizon_i):
    station_col = f"future_station_{horizon_i}"
    if station_col not in snapshot_lf.collect_schema().names():
        return snapshot_lf
    lookup_renamed = lookup.rename({
        "station_name": station_col,
        "gleisanzahl": f"future_gleisanzahl_{horizon_i}",
        "geschwindigkeit": f"future_geschwindigkeit_{horizon_i}",
        "elektrifizierung": f"future_elektrifizierung_{horizon_i}",
    })
    return snapshot_lf.join(lookup_renamed.lazy(), on=station_col, how="left")

In [26]:
for label, in_path, out_path in [("train", INFRA_TRAIN_IN, INFRA_TRAIN_OUT), ("test", INFRA_TEST_IN, INFRA_TEST_OUT)]:
    if os.path.isfile(out_path):
        print(f"[checkpoint] {out_path} already exists, skipping.")
        continue

    df = pl.read_parquet(in_path)
    lf = df.lazy()
    for i in range(1, N_FUTURE_STOPS + 1):
        lf = join_infra_for_horizon(lf, infra_lookup, i)
    enriched = lf.collect(engine="streaming")

    match_rate = enriched["future_gleisanzahl_1"].is_not_null().mean() * 100
    print(f"{label}: horizon 1 infra match rate {match_rate:.1f}%")

    enriched.write_parquet(out_path)
    print(f"Saved: {out_path}")

train: horizon 1 infra match rate 95.9%
Saved: ./dbdata/train_3city_2025-11_to_2026-04_weather_infra.parquet
test: horizon 1 infra match rate 96.0%
Saved: ./dbdata/test_3city_2026-05_to_2026-07_weather_infra.parquet


## Join timetable-structure features

Headway, dwell time, and service frequency, computed directly from the already-downloaded raw monthly files.

Per (station, calendar month), this computes:
- `station_headway_min`: median gap between consecutive *scheduled* (planned, not actual) stop events at that station that month
- `station_dwell_min`: median scheduled dwell time (planned departure minus planned arrival)
- `station_freq_per_day`: scheduled stop events that month, divided by days in the month

These use planned times throughout, deliberately, since this is meant to capture timetable *design*, not realized performance; realized performance would leak into the congestion-style features computed in Stage 3.

Planned dwell time is bounded to `MAX_PLAUSIBLE_DWELL_MIN` (180 minutes): a negative or absurdly long value indicates a data issue, such as a terminus row with a stale next-day departure time, not a real dwell.

In [27]:
TIMETABLE_TRAIN_IN = INFRA_TRAIN_OUT
TIMETABLE_TEST_IN = INFRA_TEST_OUT
TIMETABLE_TRAIN_OUT = f"{LOCAL_DATA_DIR}/train_3city_{TRAIN_SPAN_LABEL}_weather_infra_timetable.parquet"
TIMETABLE_TEST_OUT = f"{LOCAL_DATA_DIR}/test_3city_{TEST_SPAN_LABEL}_weather_infra_timetable.parquet"
TIMETABLE_LOOKUP_CACHE = f"{LOCAL_DATA_DIR}/timetable_lookup.parquet"

MAX_PLAUSIBLE_DWELL_MIN = 180


def compute_station_month_stats(month):
    path = f"{LOCAL_DATA_DIR}/germany_{month}.parquet"
    print(f"Loading raw {month}...")
    df = pl.scan_parquet(path).select(["station_name", "arrival_planned_time", "departure_planned_time"])
    df = filter_to_cities(df)

    stop_events = (
        df.with_columns(
            pl.coalesce([pl.col("arrival_planned_time"), pl.col("departure_planned_time")]).alias("planned_stop_time")
        )
        .drop_nulls("planned_stop_time")
        .select(["station_name", "planned_stop_time"])
        .sort(["station_name", "planned_stop_time"])
        .collect()
    )
    stop_events = stop_events.with_columns(
        (pl.col("planned_stop_time").diff().over("station_name").dt.total_minutes()).alias("gap_min")
    )
    headway = (
        stop_events.filter(pl.col("gap_min").is_not_null() & (pl.col("gap_min") >= 0))
        .group_by("station_name")
        .agg(pl.col("gap_min").median().alias("station_headway_min"))
    )

    days_in_month = calendar.monthrange(*(int(x) for x in month.split("-")))[1]
    frequency = (
        stop_events.group_by("station_name")
        .agg(pl.len().alias("n_stops"))
        .with_columns((pl.col("n_stops") / days_in_month).alias("station_freq_per_day"))
        .select(["station_name", "station_freq_per_day"])
    )

    dwell_raw = (
        df.drop_nulls(["arrival_planned_time", "departure_planned_time"])
        .with_columns(
            (pl.col("departure_planned_time") - pl.col("arrival_planned_time")).dt.total_minutes().alias("dwell_min")
        )
        .collect()
    )
    dwell_valid = dwell_raw.filter((pl.col("dwell_min") >= 0) & (pl.col("dwell_min") <= MAX_PLAUSIBLE_DWELL_MIN))
    print(f"  Dwell: dropped {len(dwell_raw) - len(dwell_valid):,}/{len(dwell_raw):,} rows outside "
          f"[0, {MAX_PLAUSIBLE_DWELL_MIN}] min")
    dwell = dwell_valid.group_by("station_name").agg(pl.col("dwell_min").median().alias("station_dwell_min"))

    stats = headway.join(frequency, on="station_name", how="outer_coalesce").join(dwell, on="station_name", how="outer_coalesce")
    stats = stats.with_columns(pl.lit(month).alias("year_month"))
    print(f"  {month}: {len(stats):,} stations with timetable stats")
    return stats

In [28]:
if os.path.isfile(TIMETABLE_LOOKUP_CACHE):
    print(f"[checkpoint] {TIMETABLE_LOOKUP_CACHE} already exists, loading from cache.")
    timetable_lookup = pl.read_parquet(TIMETABLE_LOOKUP_CACHE)
else:
    all_stats = [compute_station_month_stats(m) for m in ALL_MONTHS]
    timetable_lookup = pl.concat(all_stats, how="vertical_relaxed")
    timetable_lookup.write_parquet(TIMETABLE_LOOKUP_CACHE)

Loading raw 2025-11...
  Dwell: dropped 0/2,124,800 rows outside [0, 180] min
  2025-11: 184 stations with timetable stats
Loading raw 2025-12...


C:\Users\Ranjith Panicker\AppData\Local\Temp\ipykernel_23796\4096859570.py:54: DeprecationWarning: use of `how='outer_coalesce'` should be replaced with `how='full', coalesce=True`.
(Deprecated in version 0.20.29)
  stats = headway.join(frequency, on="station_name", how="outer_coalesce").join(dwell, on="station_name", how="outer_coalesce")


  Dwell: dropped 0/2,407,916 rows outside [0, 180] min
  2025-12: 184 stations with timetable stats
Loading raw 2026-01...
  Dwell: dropped 0/2,450,389 rows outside [0, 180] min
  2026-01: 184 stations with timetable stats
Loading raw 2026-02...
  Dwell: dropped 0/2,143,499 rows outside [0, 180] min
  2026-02: 184 stations with timetable stats
Loading raw 2026-03...
  Dwell: dropped 0/2,389,350 rows outside [0, 180] min
  2026-03: 184 stations with timetable stats
Loading raw 2026-04...
  Dwell: dropped 0/2,151,858 rows outside [0, 180] min
  2026-04: 184 stations with timetable stats
Loading raw 2026-05...
  Dwell: dropped 0/2,211,723 rows outside [0, 180] min
  2026-05: 184 stations with timetable stats
Loading raw 2026-06...
  Dwell: dropped 0/2,280,991 rows outside [0, 180] min
  2026-06: 184 stations with timetable stats
Loading raw 2026-07...
  Dwell: dropped 0/2,159,066 rows outside [0, 180] min
  2026-07: 184 stations with timetable stats


In [29]:
def join_timetable_for_horizon(snapshot_lf, lookup, horizon_i):
    station_col = f"future_station_{horizon_i}"
    minutes_col = f"future_minutes_ahead_{horizon_i}"
    schema_names = snapshot_lf.collect_schema().names()
    if station_col not in schema_names or minutes_col not in schema_names:
        return snapshot_lf

    ym_col = f"_target_ym_{horizon_i}"
    snapshot_lf = snapshot_lf.with_columns(
        (pl.col("snapshot_time") + pl.duration(minutes=pl.col(minutes_col))).dt.strftime("%Y-%m").alias(ym_col)
    )
    lookup_renamed = lookup.rename({
        "station_name": station_col, "year_month": ym_col,
        "station_headway_min": f"future_station_headway_min_{horizon_i}",
        "station_dwell_min": f"future_station_dwell_min_{horizon_i}",
        "station_freq_per_day": f"future_station_freq_per_day_{horizon_i}",
    })
    joined = snapshot_lf.join(lookup_renamed.lazy(), on=[station_col, ym_col], how="left")
    return joined.drop(ym_col)

In [30]:
for label, in_path, out_path in [("train", TIMETABLE_TRAIN_IN, TIMETABLE_TRAIN_OUT), ("test", TIMETABLE_TEST_IN, TIMETABLE_TEST_OUT)]:
    if os.path.isfile(out_path):
        print(f"[checkpoint] {out_path} already exists, skipping.")
        continue

    df = pl.read_parquet(in_path)
    lf = df.lazy()
    for i in range(1, N_FUTURE_STOPS + 1):
        lf = join_timetable_for_horizon(lf, timetable_lookup, i)
    enriched = lf.collect(engine="streaming")

    match_rate = enriched["future_station_headway_min_1"].is_not_null().mean() * 100
    print(f"{label}: horizon 1 timetable match rate {match_rate:.1f}%")

    enriched.write_parquet(out_path)
    print(f"Saved: {out_path}")

train: horizon 1 timetable match rate 95.9%
Saved: ./dbdata/train_3city_2025-11_to_2026-04_weather_infra_timetable.parquet
test: horizon 1 timetable match rate 96.0%
Saved: ./dbdata/test_3city_2026-05_to_2026-07_weather_infra_timetable.parquet


## Final step: exclude rail-replacement and rare train types

Two categories of `train_type` are excluded, identically from train and test:

1. **`Bus`** (Schienenersatzverkehr, rail-replacement bus substitutions). Excluded because a bus's delay is governed by road traffic, not track and signaling, a different physical process than the one this benchmark models, and because leaving bus-substituted trips in would let their route contribute station-adjacency edges to a graph neural network's station graph.
2. **Train types with fewer than 100 occurrences in the training data specifically**. Too rare for any model to learn a meaningful representation.

In [31]:
TRAIN_FINAL_OUT = f"{LOCAL_DATA_DIR}/train_3city_{TRAIN_SPAN_LABEL}_final.parquet"
TEST_FINAL_OUT = f"{LOCAL_DATA_DIR}/test_3city_{TEST_SPAN_LABEL}_final.parquet"

MIN_TRAIN_OCCURRENCES = 100

pre_exclusion_train = pl.read_parquet(TIMETABLE_TRAIN_OUT)
pre_exclusion_test = pl.read_parquet(TIMETABLE_TEST_OUT)
print(f"Loaded: train {len(pre_exclusion_train):,} rows, test {len(pre_exclusion_test):,} rows")

Loaded: train 2,513,797 rows, test 258,560 rows


In [ ]:
n_train_before, n_test_before = len(pre_exclusion_train), len(pre_exclusion_test)

train_final = pre_exclusion_train.filter(pl.col("train_type") != "Bus")
test_final = pre_exclusion_test.filter(pl.col("train_type") != "Bus")
print(f"'Bus' removed: {n_train_before - len(train_final):,} train rows, {n_test_before - len(test_final):,} test rows")

freq = train_final.group_by("train_type").agg(pl.len().alias("n")).sort("n")
rare_types = freq.filter(pl.col("n") < MIN_TRAIN_OCCURRENCES)["train_type"].to_list()

n_before_rare_train, n_before_rare_test = len(train_final), len(test_final)
if rare_types:
    train_final = train_final.filter(~pl.col("train_type").is_in(rare_types))
    test_final = test_final.filter(~pl.col("train_type").is_in(rare_types))
print(f"Rare types removed (< {MIN_TRAIN_OCCURRENCES} training occurrences): {rare_types}")
print(f"  -> {n_before_rare_train - len(train_final):,} train rows, {n_before_rare_test - len(test_final):,} test rows")

print(f"Train: {n_train_before:,} -> {len(train_final):,} rows ({100*len(train_final)/n_train_before:.2f}% kept)")
print(f"Test:  {n_test_before:,} -> {len(test_final):,} rows ({100*len(test_final)/n_test_before:.2f}% kept)")

train_final.write_parquet(TRAIN_FINAL_OUT)
test_final.write_parquet(TEST_FINAL_OUT)
print(f"\nPipeline complete.")
print(f"  {TRAIN_FINAL_OUT}")
print(f"  {TEST_FINAL_OUT}")

'Bus' removed: 53,556 train rows, 10,414 test rows
Rare types removed (< 100 training occurrences): ['MSM', 'IRE', 'R', 'ECE', 'RJX']
  -> 178 train rows, 91 test rows
Train: 2,513,797 -> 2,460,063 rows (97.86% kept)
Test:  258,560 -> 248,055 rows (95.94% kept)

Pipeline complete. Upload these two files to Kaggle for model training:
  ./dbdata/train_3city_2025-11_to_2026-04_final.parquet
  ./dbdata/test_3city_2026-05_to_2026-07_final.parquet
